# Taller 3 – Punto 2: Verdadera democracia

Este notebook implementa un **algoritmo genético (AG)** para repartir 50 entidades estatales entre cinco partidos políticos.

La representación es un cromosoma de 50 genes enteros. Cada posición representa una entidad y su valor (`0` a `4`) indica el partido al que se asigna. La función objetivo minimiza la diferencia entre el poder recibido por cada partido y el poder que le correspondería según su proporción de curules.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42  # Cambiar por None si se desea una corrida diferente
rng = np.random.default_rng(SEMILLA)

partidos = ["Partido A", "Partido B", "Partido C", "Partido D", "Partido E"]

# Distribución no uniforme de referencia para generar las 50 curules
prob_curules = np.array([0.34, 0.25, 0.18, 0.13, 0.10])
curules = rng.multinomial(50, prob_curules)

tabla_curules = pd.DataFrame({
    "Partido": partidos,
    "Curules": curules,
    "Participación": curules / 50
})
tabla_curules

In [ ]:
entidades = [
"Ministerio del Interior","Ministerio de Relaciones Exteriores","Ministerio de Hacienda","Ministerio de Justicia",
"Ministerio de Defensa","Ministerio de Agricultura","Ministerio de Salud","Ministerio de Trabajo",
"Ministerio de Minas y Energía","Ministerio de Comercio","Ministerio de Educación","Ministerio de Ambiente",
"Ministerio de Vivienda","Ministerio TIC","Ministerio de Transporte","Ministerio de Cultura",
"Ministerio del Deporte","Departamento Nacional de Planeación","DIAN","DANE","ICBF","SENA","Invías","ANI",
"Aeronáutica Civil","Superintendencia de Industria y Comercio","Superintendencia Financiera","Superintendencia de Salud",
"Unidad Nacional de Gestión del Riesgo","Agencia Nacional de Tierras","Agencia de Desarrollo Rural",
"Agencia Nacional de Minería","Agencia Nacional de Hidrocarburos","Agencia Nacional del Espectro",
"Agencia Nacional de Seguridad Vial","Agencia Nacional Inmobiliaria","Minciencias","ICETEX",
"Unidad de Restitución de Tierras","Unidad para las Víctimas","Fondo Adaptación","Prosperidad Social",
"Instituto Nacional de Salud","INVIMA","IDEAM","IGAC","Archivo General de la Nación",
"Instituto Colombiano de Antropología e Historia","RTVC","Colombia Compra Eficiente"
]

pesos = rng.integers(1, 101, size=len(entidades))

tabla_entidades = pd.DataFrame({
    "Entidad": entidades,
    "Peso político": pesos
})
tabla_entidades

## Definición de la función de aptitud

Sea \(P_T\) el poder total disponible. El poder objetivo de cada partido se calcula como:

\[
P_i^{obj}=P_T\frac{C_i}{50}
\]

donde \(C_i\) es el número de curules del partido \(i\).

Para una asignación candidata se calcula el poder real \(P_i^{real}\). El error normalizado es:

\[
E=\frac{\sum_i |P_i^{real}-P_i^{obj}|}{P_T}
\]

El AG busca **minimizar** este error. Una forma equivalente de expresarlo como aptitud es:

\[
f=\frac{1}{1+E}
\]

Cuanto más cercana a 1 sea la aptitud, mejor es la solución.

In [ ]:
poder_total = pesos.sum()
objetivo = (curules / 50) * poder_total

def poder_por_partido(individuo):
    return np.bincount(individuo, weights=pesos, minlength=5)

def error(individuo):
    poder = poder_por_partido(individuo)
    return np.sum(np.abs(poder - objetivo)) / poder_total

def aptitud(individuo):
    return 1 / (1 + error(individuo))

In [ ]:
# Parámetros del AG
TAM_POB = 300
GENERACIONES = 400
PROB_MUT = 0.04
ELITE = 4
TAM_TORNEO = 4

# Población inicial
poblacion = rng.integers(0, 5, size=(TAM_POB, len(entidades)))

def seleccion_torneo(poblacion):
    idx = rng.choice(len(poblacion), TAM_TORNEO, replace=False)
    candidatos = poblacion[idx]
    errores = np.array([error(ind) for ind in candidatos])
    return candidatos[np.argmin(errores)].copy()

def cruce_uniforme(padre1, padre2):
    mascara = rng.random(len(entidades)) < 0.5
    hijo1 = np.where(mascara, padre1, padre2)
    hijo2 = np.where(mascara, padre2, padre1)
    return hijo1, hijo2

def mutar(individuo):
    mascara = rng.random(len(entidades)) < PROB_MUT
    individuo[mascara] = rng.integers(0, 5, size=mascara.sum())
    return individuo

historial = []

for gen in range(GENERACIONES):
    errores = np.array([error(ind) for ind in poblacion])
    orden = np.argsort(errores)
    historial.append(errores[orden[0]])

    nueva = [poblacion[i].copy() for i in orden[:ELITE]]

    while len(nueva) < TAM_POB:
        p1 = seleccion_torneo(poblacion)
        p2 = seleccion_torneo(poblacion)
        h1, h2 = cruce_uniforme(p1, p2)

        nueva.append(mutar(h1))
        if len(nueva) < TAM_POB:
            nueva.append(mutar(h2))

    poblacion = np.array(nueva)

errores_finales = np.array([error(ind) for ind in poblacion])
mejor = poblacion[np.argmin(errores_finales)]
mejor_error = errores_finales.min()
mejor_aptitud = aptitud(mejor)

print(f"Mejor error normalizado: {mejor_error:.6f}")
print(f"Mejor aptitud: {mejor_aptitud:.6f}")

In [ ]:
poder_real = poder_por_partido(mejor)

resumen = pd.DataFrame({
    "Partido": partidos,
    "Curules": curules,
    "% curules": curules / 50 * 100,
    "Poder objetivo": objetivo,
    "Poder asignado": poder_real,
    "Diferencia": poder_real - objetivo
})

resumen.round(2)

In [ ]:
# Matriz de poder: filas = entidades, columnas = partidos.
# Cada fila tiene el peso de la entidad únicamente en el partido al que fue asignada.
matriz_poder = pd.DataFrame(0, index=entidades, columns=partidos, dtype=int)

for i, partido_idx in enumerate(mejor):
    matriz_poder.iloc[i, partido_idx] = int(pesos[i])

matriz_poder.insert(0, "Peso político", pesos)
matriz_poder.insert(0, "Entidad", entidades)

matriz_poder

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(historial)
plt.xlabel("Generación")
plt.ylabel("Mejor error normalizado")
plt.title("Convergencia del algoritmo genético")
plt.grid(True)
plt.show()

x = np.arange(len(partidos))
w = 0.38
plt.figure(figsize=(9, 4))
plt.bar(x - w/2, objetivo, width=w, label="Objetivo")
plt.bar(x + w/2, poder_real, width=w, label="Asignado")
plt.xticks(x, partidos)
plt.ylabel("Puntos de poder")
plt.title("Poder objetivo vs. poder asignado")
plt.legend()
plt.show()

In [ ]:
# Guardar resultados por si se quieren subir al repositorio
tabla_curules.to_csv("curules_partidos.csv", index=False)
tabla_entidades.to_csv("entidades_pesos.csv", index=False)
matriz_poder.to_csv("matriz_poder.csv", index=False)
resumen.to_csv("resumen_poder.csv", index=False)

print("Archivos CSV generados correctamente.")